In [1]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.9/258.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.3/309.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.64.1
    Uninstalling grpcio-1.64.1:
      Successfully uninstalled grpcio-1.64.1
ERROR: pip's dependency resolver does not currently tak

In [2]:
import logging
from typing import List, Dict, Any
from qdrant_client import QdrantClient
from qdrant_client.models import Filter

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_d"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def get_unique_metadata():
    try:
        points = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=10000,
            with_payload=True
        )[0]

        categories, diseases, intents = set(), set(), set()

        for point in points:
            if not point.payload:
                continue
            metadata = point.payload.get('metadata', {})
            categories.add(metadata.get('카테고리', ''))
            diseases.add(metadata.get('질병', ''))
            intents.add(metadata.get('의도', ''))

        categories = sorted(filter(None, categories))
        diseases = sorted(filter(None, diseases))
        intents = sorted(filter(None, intents))

        logging.info(f"메타데이터 추출 완료: {len(categories)} 카테고리, {len(diseases)} 질병, {len(intents)} 의도")
        return categories, diseases, intents

    except Exception as e:
        logging.error(f"메타데이터 추출 중 오류 발생: {str(e)}", exc_info=True)
        return [], [], []

def analyze_query(query: str, categories: List[str], diseases: List[str], intents: List[str]) -> Dict[str, str]:
    query_lower = query.lower()

    # 카테고리 매칭
    category = next((cat for cat in categories if cat.lower() in query_lower), None)

    # 질병 매칭
    disease = next((dis for dis in diseases if dis.lower() in query_lower), None)

    # 의도 매칭
    intent_keywords = {
        '증상': ['증상', '징후', '어떻게 나타나'],
        '치료': ['치료', '처치', '어떻게 낫'],
        '예방': ['예방', '방지', '어떻게 막'],
        '원인': ['원인', '이유', '왜'],
        '진단': ['진단', '검사', '어떻게 알'],
    }

    intent = None
    for key, keywords in intent_keywords.items():
        if any(keyword in query_lower for keyword in keywords):
            intent = key
            break

    if intent not in intents:
        intent = None

    return {'카테고리': category, '질병': disease, '의도': intent}

def search_documents(query: str, filter_params: Dict[str, str] = None) -> List[Dict[str, Any]]:
    try:
        filter_conditions = []
        if filter_params:
            for key, value in filter_params.items():
                if value:
                    filter_conditions.append(
                        Filter(
                            must=[
                                Filter(
                                    key=f"metadata.{key}",
                                    match={"value": value}
                                )
                            ]
                        )
                    )

        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=[0.0] * 768,  # 임시로 0으로 채운 벡터 사용
            query_filter=Filter(
                must=filter_conditions
            ) if filter_conditions else None,
            limit=5
        )

        return [
            {
                "content": point.payload.get("답변", ""),
                "metadata": point.payload.get("metadata", {})
            }
            for point in search_result
        ]

    except Exception as e:
        logging.error(f"문서 검색 중 오류 발생: {str(e)}", exc_info=True)
        return []

# 메타데이터 가져오기
categories, diseases, intents = get_unique_metadata()
print("카테고리:", categories)
print("질병:", diseases)
print("의도:", intents)

# 사용자 입력 받기
user_query = input("질문을 입력하세요: ")

# 쿼리 분석
analyzed_metadata = analyze_query(user_query, categories, diseases, intents)
print("\n분석된 메타데이터:", analyzed_metadata)

# 검색 수행
results = search_documents(user_query, analyzed_metadata)

print("\n검색 결과:")
if results:
    for result in results:
        print(f"내용: {result['content'][:100]}...")
        print(f"메타데이터: {result['metadata']}")
        print()
else:
    print("검색 결과가 없습니다.")

카테고리: []
질병: []
의도: []
질문을 입력하세요: 응급한 고혈압 증상이 궁금해요

분석된 메타데이터: {'카테고리': None, '질병': None, '의도': None}

검색 결과:
내용: 고혈압을 진단받은 환자들은 처음 치료 전에 기본 검사를 받아야 합니다. 정기적인 검진을 통해 고혈압을 관리하기 위해 혈압 상태를 확인하고 치료 계획을 수립할 수 있습니다. 고혈압 ...
메타데이터: {}

내용: 기본 검사로는 12-유도 심전도, 소변 검사, 혈액 검사, 흉부 X-선 촬영 등을 시행합니다. 이러한 검사들은 고혈압의 정도와 심혈관 질환의 위험도를 평가하는 데 도움을 줍니다. ...
메타데이터: {}

내용: 고혈압은 일반적으로 첫 치료를 시작하기 전에 기본 검사가 이루어져야 합니다. 이는 처음 진단을 받은 환자들이 매년 기본 검사를 받아야 함을 의미합니다. 기본 검사에는 기본 심전도,...
메타데이터: {}

내용: 고혈압 진단 후 기본 검진 이외에도 고혈압으로 인한 다른 증상을 주시하는 것이 중요합니다. 예를 들어, 심한 두통, 눈 증상, 실신 등의 증상이 있는 경우 해당 증상을 주시하고 조...
메타데이터: {}

내용: 고혈압 진단을 받은 환자들은 처음 치료 전에 기본 검사를 받고, 이를 통해 치료 계획을 세우게 됩니다. 그 후 매년 기본 검사를 실시하고, 필요에 따라 추가 검사를 고려할 수 있습...
메타데이터: {}



In [6]:
from collections import defaultdict

def get_all_metadata():
    try:
        offset = 0
        limit = 1000
        all_metadata = defaultdict(set)
        total_points = 0

        while True:
            points = client.scroll(
                collection_name=COLLECTION_NAME,
                limit=limit,
                offset=offset,
                with_payload=True,
                with_vectors=False
            )[0]

            if not points:
                break

            for point in points:
                if point.payload:
                    # 각 필드에 대한 고유 값을 추가
                    if '답변' in point.payload:
                        all_metadata['답변'].add(point.payload['답변'])
                    if '부서' in point.payload:
                        all_metadata['부서'].add(point.payload['부서'])
                    if '의도' in point.payload:
                        all_metadata['의도'].add(point.payload['의도'])
                    if '질병' in point.payload:
                        all_metadata['질병'].add(point.payload['질병'])
                    if '질병 카테고리' in point.payload:
                        all_metadata['질병 카테고리'].add(point.payload['질병 카테고리'])

                total_points += 1

            offset += limit
            logging.info(f"Processed {total_points} points so far...")

        return all_metadata, total_points

    except Exception as e:
        logging.error(f"메타데이터 추출 중 오류 발생: {str(e)}", exc_info=True)
        return {}, 0

# 메타데이터 가져오기
metadata_dict, total_points = get_all_metadata()

print(f"\n총 처리된 문서 수: {total_points}")
print("\n메타데이터 필드 및 고유 값:")
for field, values in metadata_dict.items():
    print(f"\n{field}:")
    for value in sorted(values):
        print(f"  - {value}")

print("\n각 메타데이터 필드의 고유 값 개수:")
for field, values in metadata_dict.items():
    print(f"{field}: {len(values)}개")


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  - 먼저, 스트레스와 피로를 피하는 것이 중요합니다. 스트레스는 일상 생활에 부정적인 영향을 미치며, 두통의 위험을 증가시킬 수 있습니다. 또한, 충분한 휴식을 취하고 충분한 수면을 취하는 것도 두통 예방에 도움이 됩니다. 수면 부족은 두통을 유발할 수 있으므로 충분한 수면 시간을 확보하는 것이 필요합니다.
  - 먼저, 식사 중에는 너무 긴 시간동안 식사를 하지 않도록 주의해야 합니다. 음식을 먹는 동안 다른 활동을 하거나 다른 생각을 하는 것은 두통을 유발할 수 있기 때문입니다. 또한, 식사할 때는 너무 짜거나 매운 음식을 피하는 것이 좋습니다. 이는 간이 강하게 되면 뇌로 가는 혈류를 방해하고 혈액이 고이게 만들어 두통을 유발할 수 있습니다.
  - 먼저, 안지오텐신 전환효소 억제제와 안지오텐신 수용체 차단제는 안지오텐신계의 혈압 상승을 억제하고 혈관을 확장시킵니다. 이뇨제는 혈압을 낮춥니다. 칼륨 차단제는 혈관을 확장시키는데 사용되며, 칼슘채널 차단제는 혈관이 확장되는 것을 방지하는데 효과적입니다. 베타 차단제는 심근경색과 협심증이 있는 고혈압 환자에게 좋은 선택입니다. 그리고, 이뇨제는 체액량을 감소시키고 신장에서 나트륨과 수분의 재흡수를 억제하여 체액량을 감소시킵니다.
  - 먼저, 약물 치료는 비약물적인 치료와 함께 병행되어야 합니다. 환자의 생활습관과 식단 조절은 혈압을 낮추고 고혈압 예방에 중요한 역할을 합니다. 하지만 약물 치료만으로는 혈압을 정상 범위로 되돌리기 어렵기 때문에 생활습관과 식단 관리가 필요합니다. 예를 들어, 스트레스, 운동 부족, 고칼륨 식품 섭취 부족, 흡연 등의 요소를 주의해야 합니다. 또한, 체중 감량, 규칙적인 운동, 금연 등도 중요한 치료 및 예방 방법입니다. 약물 치료로 혈압이 정상 범위로 회복되지 않을 경우, 고혈압 치료 방법으로 혈액 순환 개선, 체중 감량, 염분 섭취 제한, 혈압 조절을 위해 약물 요법이 시행됩니다. 하지만 약물 치료만으로 혈

매우 잘 나오는 것을 확인할 수 있음.

In [9]:
import logging
from typing import List, Dict, Any
from qdrant_client import QdrantClient
from qdrant_client.models import Filter

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_d"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def get_all_metadata():
    try:
        offset = 0
        limit = 1000
        all_metadata = defaultdict(set)
        total_points = 0

        while True:
            points = client.scroll(
                collection_name=COLLECTION_NAME,
                limit=limit,
                offset=offset,
                with_payload=True,
                with_vectors=False
            )[0]

            if not points:
                break

            for point in points:
                if point.payload:
                    if '답변' in point.payload:
                        all_metadata['답변'].add(point.payload['답변'])
                    if '부서' in point.payload:
                        all_metadata['부서'].add(point.payload['부서'])
                    if '의도' in point.payload:
                        all_metadata['의도'].add(point.payload['의도'])
                    if '질병' in point.payload:
                        all_metadata['질병'].add(point.payload['질병'])
                    if '질병 카테고리' in point.payload:
                        all_metadata['질병 카테고리'].add(point.payload['질병 카테고리'])

                total_points += 1

            offset += limit
            logging.info(f"Processed {total_points} points so far...")

        return all_metadata, total_points

    except Exception as e:
        logging.error(f"메타데이터 추출 중 오류 발생: {str(e)}", exc_info=True)
        return {}, 0

def get_unique_metadata():
    metadata_dict, _ = get_all_metadata()
    return (
        sorted(metadata_dict.get('부서', [])),
        sorted(metadata_dict.get('의도', [])),
        sorted(metadata_dict.get('질병', [])),
        sorted(metadata_dict.get('질병 카테고리', []))
    )

def analyze_query(query: str, departments: List[str], diseases: List[str], intents: List[str], disease_categories: List[str]) -> Dict[str, str]:
    query_lower = query.lower()

    # 카테고리, 질병, 부서, 의도 매칭
    department = next((dep for dep in departments if dep.lower() in query_lower), None)
    disease = next((dis for dis in diseases if dis.lower() in query_lower), None)
    disease_category = next((cat for cat in disease_categories if cat.lower() in query_lower), None)

    intent_keywords = {
        '증상': ['증상', '징후', '어떻게 나타나'],
        '치료': ['치료', '처치', '어떻게 낫'],
        '예방': ['예방', '방지', '어떻게 막'],
        '원인': ['원인', '이유', '왜'],
        '진단': ['진단', '검사', '어떻게 알'],
    }

    intent = None
    for key, keywords in intent_keywords.items():
        if any(keyword in query_lower for keyword in keywords):
            intent = key
            break

    return {
        '부서': department,
        '질병': disease,
        '질병 카테고리': disease_category,
        '의도': intent
    }

def search_documents(query: str, filter_params: Dict[str, str] = None) -> List[Dict[str, Any]]:
    try:
        filter_conditions = []
        if filter_params:
            for key, value in filter_params.items():
                if value:
                    filter_conditions.append(
                        Filter(
                            must=[
                                {
                                    "key": f"metadata.{key}",
                                    "match": {"value": value}
                                }
                            ]
                        )
                    )

        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=[0.0] * 768,  # 임시로 0으로 채운 벡터 사용
            query_filter=Filter(
                must=filter_conditions
            ) if filter_conditions else None,
            limit=5
        )

        return [
            {
                "content": point.payload.get("답변", ""),
                "metadata": point.payload.get("metadata", {})
            }
            for point in search_result
        ]

    except Exception as e:
        logging.error(f"문서 검색 중 오류 발생: {str(e)}", exc_info=True)
        return []


# 사용자 입력 받기
user_query = input("질문을 입력하세요: ")

# 메타데이터 가져오기
departments, intents, diseases, disease_categories = get_unique_metadata()

# 쿼리 분석
analyzed_metadata = analyze_query(user_query, departments, diseases, intents, disease_categories)
print("\n분석된 메타데이터:", analyzed_metadata)

# 검색 수행
results = search_documents(user_query, analyzed_metadata)

print("\n검색 결과:")
if results:
    for result in results:
        print(f"내용: {result['content'][:100]}...")
        print(f"메타데이터: {result['metadata']}")
        print()
else:
    print("검색 결과가 없습니다.")


질문을 입력하세요: 응급한 고혈압 증상이 궁금해요

분석된 메타데이터: {'부서': None, '질병': '고혈압', '질병 카테고리': None, '의도': '증상'}

검색 결과:
검색 결과가 없습니다.


In [10]:
!pip install sentence-transformers

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [11]:
from sentence_transformers import SentenceTransformer

# 사전 훈련된 모델 로드
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def vectorize_query(query: str) -> List[float]:
    return model.encode(query).tolist()

def search_documents(query: str, filter_params: Dict[str, str] = None) -> List[Dict[str, Any]]:
    try:
        # 질문을 벡터로 변환
        query_vector = vectorize_query(query)

        filter_conditions = []
        if filter_params:
            for key, value in filter_params.items():
                if value:
                    filter_conditions.append(
                        Filter(
                            must=[
                                {
                                    "key": f"metadata.{key}",
                                    "match": {"value": value}
                                }
                            ]
                        )
                    )

        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,  # 벡터화된 질문 사용
            query_filter=Filter(
                must=filter_conditions
            ) if filter_conditions else None,
            limit=5
        )

        return [
            {
                "content": point.payload.get("답변", ""),
                "metadata": point.payload.get("metadata", {})
            }
            for point in search_result
        ]

    except Exception as e:
        logging.error(f"문서 검색 중 오류 발생: {str(e)}", exc_info=True)
        return []

# 사용자 입력 받기
user_query = input("질문을 입력하세요: ")

# 메타데이터 가져오기
departments, intents, diseases, disease_categories = get_unique_metadata()

# 쿼리 분석
analyzed_metadata = analyze_query(user_query, departments, diseases, intents, disease_categories)
print("\n분석된 메타데이터:", analyzed_metadata)

# 검색 수행
results = search_documents(user_query, analyzed_metadata)

print("\n검색 결과:")
if results:
    for result in results:
        print(f"내용: {result['content'][:100]}...")
        print(f"메타데이터: {result['metadata']}")
        print()
else:
    print("검색 결과가 없습니다.")


/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

질문을 입력하세요: 응급한 고혈압 증상이 궁금해요

분석된 메타데이터: {'부서': None, '질병': '고혈압', '질병 카테고리': None, '의도': '증상'}


ERROR:root:문서 검색 중 오류 발생: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Wrong input: Vector dimension error: expected dim: 768, got 384"},"time":0.000494524}'
Traceback (most recent call last):
  File "<ipython-input-11-e36eb715323b>", line 29, in search_documents
    search_result = client.search(
  File "/usr/local/lib/python3.10/dist-packages/qdrant_client/qdrant_client.py", line 372, in search
    return self._client.search(
  File "/usr/local/lib/python3.10/dist-packages/qdrant_client/qdrant_remote.py", line 553, in search
    search_result = self.http.points_api.search_points(
  File "/usr/local/lib/python3.10/dist-packages/qdrant_client/http/api/points_api.py", line 1616, in search_points
    return self._build_for_search_points(
  File "/usr/local/lib/python3.10/dist-packages/qdrant_client/http/api/points_api.py", line 750, in _build_for_search_points
    return self.api_client.request(
  File "/usr/local/lib/python3.10/dist-packages/qdrant_


검색 결과:
검색 결과가 없습니다.


 오류 : Qdrant는 768차원 벡터를 기대하고 있지만, 현재 384차원 벡터가 전달됨.

모델을 원래 것으로 맞추면 해결될 것으로 사료됨